# 2. Model Training & Hyperparameter Tuning
Models compared:
- **Logistic Regression** — interpretable baseline, no tuning
- **XGBoost** — tuned with Optuna (100 trials), `scale_pos_weight` tuned as a hyperparameter, optimized for PR-AUC
- **Random Forest** — `class_weight='balanced_subsample'`, tuned with Optuna (100 trials)
- **LightGBM** — `is_unbalance=True` + tuned `scale_pos_weight`, tuned with Optuna (100 trials)

Primary metric: **PR-AUC** (average precision) — more sensitive to minority-class performance than ROC-AUC under ~9% positive-class imbalance.
Precision/Recall/F1 for each model are reported at that model's own F1-optimal threshold (not the default 0.5), since raw probabilities cluster well below 0.5 for all models here.

> Temporal split: train 2015-2022, test 2023-2025 (no data leakage).


In [1]:
import joblib
import numpy as np
import pandas as pd
import warnings
import optuna
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    classification_report, recall_score, precision_score, f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import mlflow
import mlflow.sklearn

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('njit_dropout_prediction')

OUTPUT_DIR   = Path('outputs')
RANDOM_STATE = 42
CV_FOLDS     = 5
N_TRIALS     = 100

X_train = joblib.load(OUTPUT_DIR / 'X_train.pkl')
X_test  = joblib.load(OUTPUT_DIR / 'X_test.pkl')
y_train = joblib.load(OUTPUT_DIR / 'y_train.pkl')
y_test  = joblib.load(OUTPUT_DIR / 'y_test.pkl')

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.4f} | Test: {y_test.mean():.4f}")


C:\Users\prati\Desktop\Project\ARR_NJIT_RD2\ARR_NJIT_RD2\ML_Model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/07/02 08:33:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/07/02 08:33:36 INFO mlflow.store.db.utils: Updating database tables


2026/07/02 08:33:37 INFO mlflow.tracking.fluent: Experiment with name 'njit_dropout_prediction' does not exist. Creating a new experiment.


Train: (124463, 30) | Test: (65156, 30)
Train positive rate: 0.1072 | Test: 0.1065


## 2.1 Shared evaluation helper
Computes PR-AUC (threshold-independent) plus Precision/Recall/F1 at the model's own F1-optimal threshold.


In [2]:
def evaluate_model(name, model, X_test, y_test, cv_metric_value, cv_metric_std='-', stage='baseline'):
    y_prob = model.predict_proba(X_test)[:, 1]
    test_auc  = roc_auc_score(y_test, y_prob)
    pr_auc    = average_precision_score(y_test, y_prob)

    prec, rec, thresholds = precision_recall_curve(y_test, y_prob)
    f1_scores = 2 * prec * rec / (prec + rec + 1e-8)
    best_idx  = f1_scores[:-1].argmax()  # last point has no threshold
    opt_threshold = thresholds[best_idx]

    y_pred_opt = (y_prob >= opt_threshold).astype(int)
    precision  = precision_score(y_test, y_pred_opt, zero_division=0)
    recall     = recall_score(y_test, y_pred_opt, zero_division=0)
    f1         = f1_score(y_test, y_pred_opt, zero_division=0)

    print(f"{name}: CV={cv_metric_value:.4f} | Test AUC={test_auc:.4f} | PR-AUC={pr_auc:.4f} | "
          f"opt_thr={opt_threshold:.4f} | Precision={precision:.4f} | Recall={recall:.4f} | F1={f1:.4f}")

    # Log this model to MLflow: params, metrics, and the fitted model artifact
    with mlflow.start_run(run_name=name):
        raw_params = model.get_params()
        loggable_params = {k: v for k, v in raw_params.items() if isinstance(v, (str, int, float, bool)) or v is None}
        mlflow.log_params(loggable_params)
        mlflow.set_tags({"stage": stage, "algorithm": name})
        mlflow.log_metrics({
            "cv_pr_auc": float(cv_metric_value),
            "test_auc": float(test_auc),
            "pr_auc": float(pr_auc),
            "optimal_threshold": float(opt_threshold),
            "precision_at_risk": float(precision),
            "recall_at_risk": float(recall),
            "f1_at_risk": float(f1),
        })
        mlflow.sklearn.log_model(model, name="model", serialization_format="cloudpickle")

    return {
        "Model": name,
        "CV PR-AUC": round(cv_metric_value, 4),
        "CV Std": cv_metric_std if cv_metric_std == '-' else round(cv_metric_std, 4),
        "Test AUC": round(test_auc, 4),
        "PR-AUC": round(pr_auc, 4),
        "Optimal Threshold": round(float(opt_threshold), 4),
        "Precision (at-risk)": round(precision, 4),
        "Recall (at-risk)": round(recall, 4),
        "F1 (at-risk)": round(f1, 4),
    }


## 2.2 Baseline Models (default hyperparameters + class balancing)


In [3]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
pos_weight = (1 - y_train.mean()) / y_train.mean()

baseline_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced", max_iter=2000,
            solver="saga", random_state=RANDOM_STATE,
        )),
    ]),
    "XGBoost (baseline)": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        scale_pos_weight=pos_weight, eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0,
    ),
    "Random Forest (baseline)": RandomForestClassifier(
        n_estimators=300, max_depth=None, class_weight="balanced_subsample",
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "LightGBM (baseline)": LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        is_unbalance=True, random_state=RANDOM_STATE, verbosity=-1,
    ),
}

results = []
fitted_models = {}

for name, model in baseline_models.items():
    cv_scores = cross_val_score(
        model, X_train, y_train, cv=cv, scoring="average_precision", n_jobs=-1
    )
    model.fit(X_train, y_train)
    results.append(evaluate_model(name, model, X_test, y_test, cv_scores.mean(), cv_scores.std(), stage="baseline"))
    fitted_models[name] = model

pd.DataFrame(results)


Logistic Regression: CV=0.1274 | Test AUC=0.5729 | PR-AUC=0.1271 | opt_thr=0.4886 | Precision=0.1269 | Recall=0.7838 | F1=0.2184


2026/07/02 08:37:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/07/02 08:37:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost (baseline): CV=0.1305 | Test AUC=0.6001 | PR-AUC=0.1349 | opt_thr=0.4682 | Precision=0.1342 | Recall=0.7275 | F1=0.2266


2026/07/02 08:38:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest (baseline): CV=0.1188 | Test AUC=0.5693 | PR-AUC=0.1252 | opt_thr=0.0663 | Precision=0.1221 | Recall=0.7849 | F1=0.2113


2026/07/02 08:38:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LightGBM (baseline): CV=0.1305 | Test AUC=0.6010 | PR-AUC=0.1359 | opt_thr=0.4508 | Precision=0.1336 | Recall=0.7696 | F1=0.2277


,Model,CV PR-AUC,CV Std,Test AUC,PR-AUC,Optimal Threshold,Precision (at-risk),Recall (at-risk),F1 (at-risk)
0,Logistic Regression,0.1274,0.0022,0.5729,0.1271,0.4886,0.1269,0.7838,0.2184
1,XGBoost (baseline),0.1305,0.0015,0.6001,0.1349,0.4682,0.1342,0.7275,0.2266
2,Random Forest (baseline),0.1188,0.0010,0.5693,0.1252,0.0663,0.1221,0.7849,0.2113
3,LightGBM (baseline),0.1305,0.0018,0.6010,0.1359,0.4508,0.1336,0.7696,0.2277


## 2.3 Hyperparameter Tuning with Optuna
Logistic Regression is kept as an interpretable baseline with no tuning. XGBoost, Random Forest, and LightGBM are each tuned with 100 Bayesian trials, optimizing **PR-AUC** (`average_precision`) via 3-fold CV. `scale_pos_weight` is tuned as a hyperparameter (not fixed) for the gradient-boosted models.


### 2.3.1 XGBoost


In [4]:
def objective_xgb(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 600),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 5.0, 20.0),
        "eval_metric":      "aucpr",
        "random_state":     RANDOM_STATE,
        "verbosity":        0,
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
        scoring="average_precision", n_jobs=-1,
    )
    return scores.mean()

print("Tuning XGBoost with Optuna (100 trials, optimizing PR-AUC) ...")
study_xgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)
print(f"Best CV PR-AUC: {study_xgb.best_value:.4f}")
print(f"Best params: {study_xgb.best_params}")


Tuning XGBoost with Optuna (100 trials, optimizing PR-AUC) ...



  0%|          | 0/100 [00:00<?, ?it/s]


Best trial: 0. Best value: 0.119659:   0%|          | 0/100 [00:06<?, ?it/s]


Best trial: 0. Best value: 0.119659:   1%|          | 1/100 [00:06<10:45,  6.52s/it]


Best trial: 1. Best value: 0.133569:   1%|          | 1/100 [00:13<10:45,  6.52s/it]

...


Best trial: 91. Best value: 0.137477:  99%|█████████▉| 99/100 [05:11<00:02,  2.23s/it]


Best trial: 99. Best value: 0.137521:  99%|█████████▉| 99/100 [05:13<00:02,  2.23s/it]


Best trial: 99. Best value: 0.137521: 100%|██████████| 100/100 [05:13<00:00,  2.32s/it]


Best trial: 99. Best value: 0.137521: 100%|██████████| 100/100 [05:13<00:00,  3.14s/it]

Best CV PR-AUC: 0.1375
Best params: {'n_estimators': 152, 'max_depth': 5, 'learning_rate': 0.01730280048834504, 'subsample': 0.8351869467439949, 'colsample_bytree': 0.8219142667790593, 'min_child_weight': 18, 'gamma': 4.309488925030828, 'scale_pos_weight': 11.21167556343957}


In [5]:
tuned_xgb = XGBClassifier(**study_xgb.best_params, eval_metric="aucpr", random_state=RANDOM_STATE, verbosity=0)
tuned_xgb.fit(X_train, y_train)

tuned_models = {}
tune_results = []

tuned_models["XGBoost (tuned)"] = tuned_xgb
tune_results.append(evaluate_model("XGBoost (tuned)", tuned_xgb, X_test, y_test, study_xgb.best_value, stage="tuned"))


2026/07/02 08:43:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost (tuned): CV=0.1375 | Test AUC=0.6107 | PR-AUC=0.1396 | opt_thr=0.5700 | Precision=0.1352 | Recall=0.7969 | F1=0.2312


### 2.3.2 Random Forest


In [6]:
def objective_rf(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 600),
        "max_depth":         trial.suggest_int("max_depth", 4, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
        "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 30),
        "max_features":      trial.suggest_float("max_features", 0.3, 1.0),
        "class_weight":      "balanced_subsample",
        "random_state":      RANDOM_STATE,
        "n_jobs":            -1,
    }
    model = RandomForestClassifier(**params)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
        scoring="average_precision", n_jobs=-1,
    )
    return scores.mean()

print("Tuning Random Forest with Optuna (100 trials, optimizing PR-AUC) ...")
study_rf = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_rf.optimize(objective_rf, n_trials=N_TRIALS, show_progress_bar=True)
print(f"Best CV PR-AUC: {study_rf.best_value:.4f}")
print(f"Best params: {study_rf.best_params}")


Tuning Random Forest with Optuna (100 trials, optimizing PR-AUC) ...



  0%|          | 0/100 [00:00<?, ?it/s]


Best trial: 0. Best value: 0.129898:   0%|          | 0/100 [00:29<?, ?it/s]


Best trial: 0. Best value: 0.129898:   1%|          | 1/100 [00:29<48:25, 29.35s/it]


Best trial: 1. Best value: 0.136474:   1%|          | 1/100 [00:40<48:25, 29.35s/it]

...


Best trial: 56. Best value: 0.136828:  99%|█████████▉| 99/100 [42:07<00:31, 31.14s/it]


Best trial: 56. Best value: 0.136828:  99%|█████████▉| 99/100 [42:38<00:31, 31.14s/it]


Best trial: 56. Best value: 0.136828: 100%|██████████| 100/100 [42:38<00:00, 31.15s/it]


Best trial: 56. Best value: 0.136828: 100%|██████████| 100/100 [42:38<00:00, 25.58s/it]

Best CV PR-AUC: 0.1368
Best params: {'n_estimators': 240, 'max_depth': 7, 'min_samples_split': 15, 'min_samples_leaf': 30, 'max_features': 0.9014004499270568}


In [7]:
tuned_rf = RandomForestClassifier(
    **study_rf.best_params, class_weight="balanced_subsample",
    random_state=RANDOM_STATE, n_jobs=-1,
)
tuned_rf.fit(X_train, y_train)

tuned_models["Random Forest (tuned)"] = tuned_rf
tune_results.append(evaluate_model("Random Forest (tuned)", tuned_rf, X_test, y_test, study_rf.best_value, stage="tuned"))


Random Forest (tuned): CV=0.1368 | Test AUC=0.6103 | PR-AUC=0.1399 | opt_thr=0.4708 | Precision=0.1347 | Recall=0.7976 | F1=0.2305


2026/07/02 09:26:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


### 2.3.3 LightGBM


In [8]:
def objective_lgbm(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 600),
        "max_depth":        trial.suggest_int("max_depth", 3, 12),
        "num_leaves":       trial.suggest_int("num_leaves", 15, 255),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 5.0, 20.0),
        "random_state":     RANDOM_STATE,
        "verbosity":        -1,
    }
    model = LGBMClassifier(**params)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
        scoring="average_precision", n_jobs=-1,
    )
    return scores.mean()

print("Tuning LightGBM with Optuna (100 trials, optimizing PR-AUC) ...")
study_lgbm = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS, show_progress_bar=True)
print(f"Best CV PR-AUC: {study_lgbm.best_value:.4f}")
print(f"Best params: {study_lgbm.best_params}")


Tuning LightGBM with Optuna (100 trials, optimizing PR-AUC) ...



  0%|          | 0/100 [00:00<?, ?it/s]


Best trial: 0. Best value: 0.123201:   0%|          | 0/100 [00:15<?, ?it/s]


Best trial: 0. Best value: 0.123201:   1%|          | 1/100 [00:15<26:17, 15.94s/it]


Best trial: 1. Best value: 0.124796:   1%|          | 1/100 [00:22<26:17, 15.94s/it]

...


Best trial: 92. Best value: 0.136001:  99%|█████████▉| 99/100 [07:13<00:03,  3.04s/it]


Best trial: 92. Best value: 0.136001:  99%|█████████▉| 99/100 [07:15<00:03,  3.04s/it]


Best trial: 92. Best value: 0.136001: 100%|██████████| 100/100 [07:15<00:00,  2.83s/it]


Best trial: 92. Best value: 0.136001: 100%|██████████| 100/100 [07:15<00:00,  4.36s/it]

Best CV PR-AUC: 0.1360
Best params: {'n_estimators': 399, 'max_depth': 3, 'num_leaves': 231, 'learning_rate': 0.016824662886276414, 'subsample': 0.947873154489602, 'colsample_bytree': 0.9459520098133083, 'min_child_samples': 78, 'scale_pos_weight': 17.75791617892676}


In [9]:
tuned_lgbm = LGBMClassifier(**study_lgbm.best_params, random_state=RANDOM_STATE, verbosity=-1)
tuned_lgbm.fit(X_train, y_train)

tuned_models["LightGBM (tuned)"] = tuned_lgbm
tune_results.append(evaluate_model("LightGBM (tuned)", tuned_lgbm, X_test, y_test, study_lgbm.best_value, stage="tuned"))


2026/07/02 09:34:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LightGBM (tuned): CV=0.1360 | Test AUC=0.6105 | PR-AUC=0.1394 | opt_thr=0.6906 | Precision=0.1358 | Recall=0.7890 | F1=0.2317


## 2.4 Model Comparison
Ranked by **PR-AUC** (threshold-independent, appropriate for this class imbalance).


In [10]:
all_results = pd.DataFrame(results + tune_results).sort_values("PR-AUC", ascending=False)
all_results.reset_index(drop=True)


,Model,CV PR-AUC,CV Std,Test AUC,PR-AUC,Optimal Threshold,Precision (at-risk),Recall (at-risk),F1 (at-risk)
0,Random Forest (tuned),0.1368,-,0.6103,0.1399,0.4708,0.1347,0.7976,0.2305
1,XGBoost (tuned),0.1375,-,0.6107,0.1396,0.5700,0.1352,0.7969,0.2312
2,LightGBM (tuned),0.1360,-,0.6105,0.1394,0.6906,0.1358,0.7890,0.2317
3,LightGBM (baseline),0.1305,0.0018,0.6010,0.1359,0.4508,0.1336,0.7696,0.2277
4,XGBoost (baseline),0.1305,0.0015,0.6001,0.1349,0.4682,0.1342,0.7275,0.2266
5,Logistic Regression,0.1274,0.0022,0.5729,0.1271,0.4886,0.1269,0.7838,0.2184
6,Random Forest (baseline),0.1188,0.001,0.5693,0.1252,0.0663,0.1221,0.7849,0.2113


In [11]:
# Pick overall best model by PR-AUC
best_row   = all_results.iloc[0]
best_name  = best_row['Model']
best_model = tuned_models.get(best_name, fitted_models.get(best_name))
best_threshold = best_row['Optimal Threshold']

print(f"Best model: {best_name}")
print(f"PR-AUC: {best_row['PR-AUC']}")
print(f"Test AUC: {best_row['Test AUC']}")
print(f"Optimal threshold: {best_threshold}")
print(f"Precision (at-risk): {best_row['Precision (at-risk)']}")
print(f"Recall (at-risk): {best_row['Recall (at-risk)']}")
print(f"F1 (at-risk): {best_row['F1 (at-risk)']}")

y_prob_best = best_model.predict_proba(X_test)[:, 1]
y_pred_best = (y_prob_best >= best_threshold).astype(int)
print("\nFull classification report (at optimal threshold):")
print(classification_report(y_test, y_pred_best,
                            target_names=['Returned (0)', 'At-Risk (1)']))


Best model: Random Forest (tuned)
PR-AUC: 0.1399
Test AUC: 0.6103
Optimal threshold: 0.4708
Precision (at-risk): 0.1347
Recall (at-risk): 0.7976
F1 (at-risk): 0.2305

Full classification report (at optimal threshold):
              precision    recall  f1-score   support

Returned (0)       0.94      0.39      0.55     58219
 At-Risk (1)       0.13      0.80      0.23      6937

    accuracy                           0.43     65156
   macro avg       0.54      0.59      0.39     65156
weighted avg       0.86      0.43      0.52     65156



In [12]:
# Save best model + comparison table + its optimal threshold
all_results.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)
joblib.dump(best_model, OUTPUT_DIR / 'best_model.pkl')
joblib.dump(best_name,  OUTPUT_DIR / 'best_model_name.pkl')
joblib.dump(float(best_threshold), OUTPUT_DIR / 'optimal_threshold.pkl')

# Save all individual models
for name, m in {**fitted_models, **tuned_models}.items():
    safe = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    joblib.dump(m, OUTPUT_DIR / f'model_{safe}.pkl')

# Mark the winning run in MLflow so it is easy to find in the UI
best_run = mlflow.search_runs(
    filter_string=f"tags.algorithm = '{best_name}'",
    order_by=['start_time DESC'],
    max_results=1,
)
if not best_run.empty:
    with mlflow.start_run(run_id=best_run.iloc[0]['run_id']):
        mlflow.set_tag('selected_as_best', 'true')
        mlflow.log_artifact(str(OUTPUT_DIR / 'model_comparison.csv'))

print("Saved: best_model.pkl, best_model_name.pkl, optimal_threshold.pkl, model_comparison.csv, all individual models")
print(f'Best model logged to MLflow. Run: mlflow ui --backend-store-uri file:./mlruns  (then open http://127.0.0.1:5000)')
print("Next: run 03_evaluate.ipynb")


Saved: best_model.pkl, best_model_name.pkl, optimal_threshold.pkl, model_comparison.csv, all individual models
Best model logged to MLflow. Run: mlflow ui --backend-store-uri file:./mlruns  (then open http://127.0.0.1:5000)
Next: run 03_evaluate.ipynb
